# Chargement des bibliothèques nécessaires pour l'évaluation des modèles de langage pré-entrainés sur le dataset MMLU

In [3]:
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, T5Tokenizer, T5ForConditionalGeneration
from huggingface_hub import login
login()

# Classe pour évaluer et comparer les performances de différents modèles de langage sur le dataset MMLU

In [11]:
class MMLUEvaluation:
    def __init__(self):
        self.dataset = load_dataset("cais/mmlu", "all")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.models = {}

    def _load_model(self, model_name, tokenizer_class, model_class):
        tokenizer = tokenizer_class.from_pretrained(model_name)
        model = model_class.from_pretrained(model_name).to(self.device)
        self.models[model_name] = {'tokenizer': tokenizer, 'model': model}

    def _evaluate_model(self, model_name, question, choices):
        if model_name not in self.models:
            tokenizer, model = self._load_model(model_name, AutoTokenizer, AutoModelForCausalLM)
        else:
            model_info = self.models[model_name]
            tokenizer, model = model_info['tokenizer'], model_info['model']

        formatted_prompt = f"{question} Choices: {', '.join(choices)}"
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(self.device)
        outputs = model.generate(**inputs, max_length=100)
        answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return answer

    def evaluate_models_on_sample(self, sample_size=5):
        results = []
        for i in range(sample_size):
            question = self.dataset["test"][i]["question"]
            choices = self.dataset["test"][i]["choices"]
            correct_answer = self.dataset["test"][i]["answer"]

            result = {
                "question": question,
                "correct_answer": correct_answer
            }

            for model_name in self.models:
                response = self._evaluate_model(model_name, question, choices)
                result[f"{model_name}_correct"] = response.strip() == correct_answer

            results.append(result)
        return results

    def get_performance_summary(self, sample_size=5):
        results = self.evaluate_models_on_sample(sample_size)
        df = pd.DataFrame(results)

        summary = {
            model_name: df[f"{model_name}_correct"].mean() * 100
            for model_name in self.models
        }

        return pd.DataFrame([summary], columns=summary.keys())

# 1. Initialisation de l'évaluation

In [13]:
evaluation = MMLUEvaluation()

# 2. Charger les modèles au besoin (ici on charge tous les modèles)

## 2.1 GPT2

In [ ]:
evaluation._load_model("gpt2", AutoTokenizer, AutoModelForCausalLM)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## 2.2 google/flan-t5-large

In [ ]:
evaluation._load_model("google/flan-t5-large", T5Tokenizer, T5ForConditionalGeneration)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## 2.3 meta-llama/Llama-3.1-8B

In [ ]:
evaluation._load_model("meta-llama/Llama-3.1-8B", AutoTokenizer, AutoModelForCausalLM)

## 2.4 mistralai/Mistral-7B-Instruct-v0.3

In [ ]:
evaluation._load_model("mistralai/Mistral-7B-Instruct-v0.3", AutoTokenizer, AutoModelForCausalLM)

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## 2.5 google/gemma-2-2b

In [7]:
evaluation._load_model("google/gemma-2-2b", AutoTokenizer, AutoModelForCausalLM)

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/481M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

# 3. Evaluation sur un échantillon de 5 questions

In [14]:
results = evaluation.evaluate_models_on_sample(sample_size=5)

# 4. Affichage des résultats

In [15]:
for result in results:
    print(f"Question: {result['question']}")
    print(f"Réponse correcte: {result['correct_answer']}")
    for model_name in evaluation.models:
        correct = result[f"{model_name}_correct"]
        print(f"{model_name} a répondu correctement: {'Oui' if correct else 'Non'}")
    print("\n")

Question: Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.
Réponse correcte: 1


Question: Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the index of <p> in S_5.
Réponse correcte: 2


Question: Find all zeros in the indicated finite field of the given polynomial with coefficients in that field. x^5 + 3x^3 + x^2 + 2x in Z_5
Réponse correcte: 3


Question: Statement 1 | A factor group of a non-Abelian group is non-Abelian. Statement 2 | If K is a normal subgroup of H and H is a normal subgroup of G, then K is a normal subgroup of G.
Réponse correcte: 1


Question: Find the product of the given polynomials in the given polynomial ring. f(x) = 4x - 5, g(x) = 2x^2 - 4x + 2 in Z_8[x].
Réponse correcte: 1


